In [1]:
import os

import pandas as pd
from dotenv import load_dotenv, find_dotenv

from doc_chat.llm import create_llm
from doc_chat.rag.citation_retrieval_chain import CitationRetrievalChain
from doc_chat.rag.document_loader import DocumentLoader
from doc_chat.rag.multi_tenant_vector_store import MultiTenantVectorStore

_ = load_dotenv(find_dotenv())


In [ ]:
pd_documents = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/documents_processed.csv')
pd_documents

In [ ]:
# Load and split the documents
all_splits = []
for index, row in pd_documents.iterrows():
    doc_path = f"../../data/single_topic_rag_evaluation_dataset/processed/pdf/document_{index}.pdf"
    loader = DocumentLoader(doc_path)
    documents, splits = loader.load_and_split()
    pd_documents.loc[index, 'num_splits'] = int(len(splits))
    all_splits.append(splits)

pd_documents['num_splits'] = pd_documents['num_splits'].astype(int)
pd_documents

In [15]:
# create vector store
chroma_persist_directory = '/tmp/doc-chat-eval/vectorstore'
if not os.path.exists(chroma_persist_directory):
    os.makedirs(chroma_persist_directory)
user_id = 'single_topic_rag_evaluation'
vector_store = MultiTenantVectorStore(chroma_persist_directory=chroma_persist_directory, embedding_model='text-embedding-3-small')

Using Chroma persist directory: /tmp/doc-chat-eval/vectorstore


In [ ]:
# Index the documents
vector_store = MultiTenantVectorStore(chroma_persist_directory=chroma_persist_directory, embedding_model='text-embedding-3-small')
for index, splits in enumerate(all_splits):
      print('indexing document: ', index, 'number of splits: ', len(splits))
      # enable this to index the documents
      # vector_store.create_document_collection(user_id=user_id
      #                                         collection_id=f'document_{index}',
      #                                         document_splits=splits,
      #                                         file_name=f'document_{index}')

In [11]:
# create chain for retrieval
retrieval_chain = CitationRetrievalChain(retriever=None,
                                         # TODO: use vector store as retriever
                                         llm=create_llm())

In [16]:
# create function to retrieve documents and answer questions with the chain
def retrieve_and_answer(data):
    for index, question_row in data.iterrows():
        question = question_row['question']
        document_index = question_row['document_index']
        collection_id = f'document_{document_index}'
        print(f'Processing question: {index}, collection_id: {collection_id}')
        found_documents = vector_store.retrieve_documents(user_id=user_id, collection_id=collection_id, query=question, k=5)
        response = retrieval_chain.invoke(query=question, documents=found_documents)
        data.loc[index, 'answer_predicted'] = response['answer']

    return data



In [17]:
# single passage questions
pd_single_passage = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/single_passage_answer_questions.csv')
pd_result = retrieve_and_answer(pd_single_passage)
pd_result.to_csv('../../data/single_topic_rag_evaluation_dataset/results/single_passage_answer_questions_predict.csv', index=False)

Processing question: 0, collection_id: document_0
Processing question: 1, collection_id: document_0


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 2, collection_id: document_1
Processing question: 3, collection_id: document_1


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 4, collection_id: document_2
Processing question: 5, collection_id: document_2
Processing question: 6, collection_id: document_3
Processing question: 7, collection_id: document_3
Processing question: 8, collection_id: document_4
Processing question: 9, collection_id: document_4
Processing question: 10, collection_id: document_5
Processing question: 11, collection_id: document_5
Processing question: 12, collection_id: document_6
Processing question: 13, collection_id: document_6
Processing question: 14, collection_id: document_7
Processing question: 15, collection_id: document_7
Processing question: 16, collection_id: document_8
Processing question: 17, collection_id: document_8
Processing question: 18, collection_id: document_9
Processing question: 19, collection_id: document_9
Processing question: 20, collection_id: document_10
Processing question: 21, collection_id: document_10
Processing question: 22, collection_id: document_11
Processing question: 23, collectio

In [18]:
# multi passage questions
pd_multi_passage = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/multi_passage_answer_questions.csv')
pd_result = retrieve_and_answer(pd_multi_passage)
pd_multi_passage.to_csv('../../data/single_topic_rag_evaluation_dataset/results/multi_passage_answer_questions_predict.csv', index=False)

Processing question: 0, collection_id: document_0
Processing question: 1, collection_id: document_0
Processing question: 2, collection_id: document_1


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 3, collection_id: document_1


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 4, collection_id: document_2
Processing question: 5, collection_id: document_2
Processing question: 6, collection_id: document_3
Processing question: 7, collection_id: document_3
Processing question: 8, collection_id: document_4
Processing question: 9, collection_id: document_4
Processing question: 10, collection_id: document_5
Processing question: 11, collection_id: document_5
Processing question: 12, collection_id: document_6
Processing question: 13, collection_id: document_6
Processing question: 14, collection_id: document_7
Processing question: 15, collection_id: document_7
Processing question: 16, collection_id: document_8
Processing question: 17, collection_id: document_8
Processing question: 18, collection_id: document_9
Processing question: 19, collection_id: document_9
Processing question: 20, collection_id: document_10
Processing question: 21, collection_id: document_10
Processing question: 22, collection_id: document_11
Processing question: 23, collectio

In [19]:
# no answer questions
pd_no_answer = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/no_answer_questions.csv')
pd_result = retrieve_and_answer(pd_no_answer)
pd_no_answer.to_csv('../../data/single_topic_rag_evaluation_dataset/results/no_answer_questions_predict.csv', index=False)

Processing question: 0, collection_id: document_0
Processing question: 1, collection_id: document_0
Processing question: 2, collection_id: document_1


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4
Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 3, collection_id: document_1
Processing question: 4, collection_id: document_2
Processing question: 5, collection_id: document_2
Processing question: 6, collection_id: document_3
Processing question: 7, collection_id: document_3
Processing question: 8, collection_id: document_4
Processing question: 9, collection_id: document_4
Processing question: 10, collection_id: document_5
Processing question: 11, collection_id: document_5
Processing question: 12, collection_id: document_6
Processing question: 13, collection_id: document_6
Processing question: 14, collection_id: document_7
Processing question: 15, collection_id: document_7
Processing question: 16, collection_id: document_8
Processing question: 17, collection_id: document_8
Processing question: 18, collection_id: document_9
Processing question: 19, collection_id: document_9
Processing question: 20, collection_id: document_10
Processing question: 21, collection_id: document_10
Processing question: 22, collection_